# Exercise 11 : Integration Case

Answer the question of the slides : how many road deaths per million inhabitants did 33 European
countries record in 2024, per age group? Steps 1–12 follow the twelve steps on the slides ; try
each step yourself before you look back. Step 13 audits a flawed analysis.

Read `datasets/eurostat-road-safety-README.md` first. The files are unchanged Eurostat extracts,
retrieved on 10 October 2026 : the road deaths of 2023 and 2024 (`tran_sf_roadus`) and the
population on 1 January 2024 (`demo_pjan`).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

DATA = Path.cwd().parents[1] / 'datasets'

## Step 1 : Raw files

Read the death file without options. Look at its shape, its first rows and the distinct values of
`unit`, `age` and `geo`. Which rows must you leave out to count each death once? Look at the
population file in the same way : which age labels does it use?

In [ ]:
# Your code here


## Step 2 : Constants

The notebook defines `AGE_CODES`, which maps each Eurostat age code to a label, `LABELS` and `BINS`,
as on the slide. Print `LABELS`. `BINS` has seven edges and makes six population bands : which
label has no band?

In [ ]:
AGE_CODES = {'Y_LT15': 'Under 15', 'Y15-17': '15-17', 'Y18-24': '18-24',
             'Y25-49': '25-49', 'Y50-64': '50-64', 'Y_GE65': '65+',
             'UNK': 'Unknown age'}
LABELS = list(AGE_CODES.values())
BINS = [0, 15, 18, 25, 50, 65, float('inf')]

In [ ]:
# Your code here


## Step 3 : Typed load

Read both files with `usecols` and with `Int64` for `TIME_PERIOD` and `OBS_VALUE`. Rename
`TIME_PERIOD`, `OBS_VALUE` and `OBS_FLAG` to `year`, `value` and `flag`. Print the row counts, the
number of empty values and the population flags.

In [ ]:
# Your code here


## Step 4 : Death counts

Keep the rows with `unit == 'NR'`, `pers_cat == 'TOTAL'`, year 2024 and `sex == 'T'`, without
`EU27_2020`. Store the sorted list of countries as `COUNTRIES`, each country's `TOTAL` row as
`totals` and the remaining age rows as `counts`. Check : 33 countries, 219 age rows and 20,875
deaths.

In [ ]:
# Your code here


## Step 5 : Age labels

Map the age codes to labels with `AGE_CODES` and keep the columns `geo`, `age_group` and `deaths`.
Check that no label is missing.

In [ ]:
# Your code here


## Step 6 : Population bands

Keep the population for `sex == 'T'` without `EU27_2020`. Leave out `TOTAL` and `UNK`, turn `Y_LT1`
into 0 and `Y_OPEN` into 100, and band the ages with `pd.cut(..., right=False)`. Sum the result to
one row per country and band, in a column `population`.

In [ ]:
# Your code here


## Step 7 : Band check

Check that the bands of each country add up to its published `TOTAL` population. Check one cell by
hand : the under-15 population of the Netherlands is 2,714,040.

In [ ]:
# Your code here


## Step 8 : Grid

Build the grid of 33 countries and 7 labels with `pd.MultiIndex.from_product()`. Left-merge the
bands and then the counts onto it, with `validate='one_to_one'` and `indicator=True`. Check that the
grid has 231 cells and that the deaths of each country add up to its `totals`.

In [ ]:
# Your code here


## Step 9 : Missing counts

List the cells without a death count. Which countries are they? Why must these cells stay missing?

In [ ]:
# Your code here


## Step 10 : Rates

Build the summary per age group with the columns `countries`, `reported`, `missing`, `deaths`,
`population` and `per_million`. `population` counts only the countries that report the group.
Which age group has the highest rate, and which has the most deaths?

In [ ]:
# Your code here


## Step 11 : Comparisons

Compare your country rates with Eurostat's own rates, `unit == 'P_MHAB'` : how many of the 186
agree after rounding? Compare your deaths with `datasets/road-deaths-expected-summary.csv`, which
was computed separately with SQL.

In [ ]:
# Your code here


## Step 12 : Chart

Draw one bar per age group with the rate per million. Label each bar with its deaths and
population, and name what the chart leaves out in the axis label. Save the figure as
`road-deaths-by-age.png` before you show it.

In [ ]:
# Your code here


## Step 13 : Audit

The notebook contains an analysis in the style of an AI assistant, written by the course team. It
runs, and it has three defects :

1. Rows that are not country counts, the rates and the EU aggregate, multiply the joined cells
2. `errors='coerce'` turns the labels for babies and for people aged 100 and over into NaN, which
   removes them from the population
3. The inner join drops the deaths of unknown age and two countries, so the deaths no longer add up

Write one check per defect that fails on the proposal, without looking back at the slides. Which of
your own steps prevents each defect?

In [ ]:
raw_deaths = pd.read_csv(DATA / 'eurostat-road-deaths-2023-2024.csv').rename(
    columns={'TIME_PERIOD': 'year', 'OBS_VALUE': 'value'})
raw_population = pd.read_csv(DATA / 'eurostat-population-2024.csv').rename(
    columns={'TIME_PERIOD': 'year', 'OBS_VALUE': 'value'})
CODES = ['Y_LT15', 'Y15-17', 'Y18-24', 'Y25-49', 'Y50-64', 'Y_GE65']

rows = raw_deaths.loc[raw_deaths['year'].eq(2024) & raw_deaths['sex'].eq('T')
                      & raw_deaths['pers_cat'].eq('TOTAL') & raw_deaths['age'].ne('TOTAL')]
years = pd.to_numeric(raw_population['age'].str[1:], errors='coerce')
bands = raw_population.assign(age=pd.cut(years, BINS, right=False, labels=CODES))
bands = (bands.loc[bands['sex'].eq('T')]
         .groupby(['geo', 'age'], observed=True, as_index=False)['value'].sum())
joined = rows.merge(bands.astype({'age': 'str'}), on=['geo', 'age'],
                    suffixes=('', '_population'))
answer = joined.groupby('age').agg(deaths=('value', 'sum'),
                                   population=('value_population', 'sum'))
answer['per_million'] = (answer['deaths'] / answer['population'] * 1_000_000).round(1)
print(answer)

In [ ]:
# Your code here


## If time permits :

1. Collect Steps 4–10 into a function `rates_for(sex)` ; the notebook gives its outline. Compute
   the rates for `'F'` and `'M'` and put the two `per_million` columns side by side. In which age
   group is the difference between women and men largest?
2. Write 6–10 sentences in the answer cell : counts and denominators, missing values and zeros,
   unknown ages, the defects of the proposal, what a rate per inhabitant cannot show, such as
   exposure and causes, and how another analyst reruns your notebook. If all 183 deaths of unknown
   age were in the 65+ group, would its rate overtake that of the 18–24 group?

In [ ]:
def rates_for(sex):
    # Steps 4-10 for one sex: return the rate per million per age group
    ...

Your answer here.